# Notebook 01 — Data Loading and Schema Validation

**Purpose:** Load all CSV scenario files from `EFD_T0001-T0003_Data/`, build the metadata table, parse each CSV into the four typed DataFrames (COMM, RATES, EXCOMM, CORR), validate schema integrity, and save `outputs/metadata.csv` and `outputs/validation_report.csv`.

**Key constraint:** `faultLocation` and `faultResistance` are stored in the metadata table **only** — they never enter the prediction path.

In [ ]:
import os
import re
import io
import warnings
from pathlib import Path


import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

## 1. Configuration

In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(os.getcwd()).parent
DATA_ROOT    = PROJECT_ROOT / 'EFD_T0001-T0003_Data'
OUTPUTS_DIR  = PROJECT_ROOT / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data root    : {DATA_ROOT}')
print(f'Outputs dir  : {OUTPUTS_DIR}')

# ── Schema definitions ─────────────────────────────────────────────────────────
COMM_COLS = [
    'timestamp', 'message_type', 'remote_id',
    'response_rate', 'num_attempts', 'num_failures',
    'ds_avg_repeater_depth', 'us_avg_repeater_depth',
    'ds_current_repeater_depth', 'us_current_repeater_depth',
    'avg_receive_signal_strength', 'incorrect_count'
]  # 12 columns

RATES_COLS = [
    'timestamp', 'message_type', 'remote_id',
    'dsRawRate0', 'dsRawRate1', 'dsRawRate2', 'dsRawRate3',
    'dsRawRate4', 'dsRawRate5', 'dsRawRate6', 'dsRawRate7',
    'usRawRate0', 'usRawRate1', 'usRawRate2', 'usRawRate3',
    'usRawRate4', 'usRawRate5', 'usRawRate6', 'usRawRate7'
]  # 19 columns

EXCOMM_COLS = [
    'timestamp', 'message_type', 'remote_id',
    'peakInputLevel', 'overflowStatus', 'berCount',
    'phaseNoise_dB', 'rxCrcFailCount'
]  # 8 columns

CORR_COLS = [
    'timestamp', 'message_type', 'remote_id',
    'peakCorrLevel_dB', 'peakUncorrLevel_dB',
    'ftryCorrTrigCount', 'userCorrTrigCount',
    'corrFired', 'ftryCorrFired'
]  # 9 columns

SCHEMA_MAP = {
    'COMM'   : COMM_COLS,
    'RATES'  : RATES_COLS,
    'EXCOMM' : EXCOMM_COLS,
    'CORR'   : CORR_COLS,
}

# Expected approximate row counts per fixture over ~15 min
EXPECTED_COUNTS = {'COMM': 92, 'RATES': 92, 'EXCOMM': 20, 'CORR': 20}

# Valid fixture ID range
REMOTE_ID_MIN = 1001
REMOTE_ID_MAX = 1090
EXPECTED_FIXTURES = set(range(REMOTE_ID_MIN, REMOTE_ID_MAX + 1))
NUM_FIXTURES = len(EXPECTED_FIXTURES)  # 90

# Expected scenario duration (seconds)
EXPECTED_DURATION_SEC = 15 * 60  # 900 s
DURATION_WARN_TOLERANCE = 0.30   # warn if duration differs by > 30 %

# Resistance code → ohms lookup
RESISTANCE_OHMS = {0: 0, 1: 5_000, 2: 14_300, 3: 19_300,
                   4: 30_000, 5: 35_000, 8: 50_000}

print('Configuration loaded.')

Project root : /Users/sadiazaman/Documents/EFD Challange
Data root    : /Users/sadiazaman/Documents/EFD Challange/EFD_T0001-T0003_Data
Outputs dir  : /Users/sadiazaman/Documents/EFD Challange/outputs
Configuration loaded.


## 2. Filename Parser — Metadata Extraction

In [4]:
FILENAME_RE = re.compile(
    r'^(?P<epoch>\d+)-(?P<commCh>\d+)-(?P<powerStep>\d+)'
    r'-(?P<faultLocation>\d+)-(?P<faultResistance>\d+)\.csv$'
)

def parse_filename(file_path: Path, run_id: str) -> dict | None:
    """Extract metadata from a CSV filename. Returns None if unparseable."""
    m = FILENAME_RE.match(file_path.name)
    if m is None:
        print(f'  [WARN] Unparseable filename: {file_path.name}')
        return None
    d = m.groupdict()
    res_code = int(d['faultResistance'])
    return {
        'file_path'       : str(file_path),
        'file_name'       : file_path.name,
        'run_id'          : run_id,
        'epoch'           : int(d['epoch']),
        'commCh'          : int(d['commCh']),
        'powerStep'       : int(d['powerStep']),
        'faultLocation'   : int(d['faultLocation']),
        'faultResistance' : res_code,
        'resistance_ohms' : RESISTANCE_OHMS.get(res_code, None),
    }


def collect_metadata() -> pd.DataFrame:
    """Walk DATA_ROOT and build the metadata table (one row per CSV)."""
    records = []
    for run_dir in sorted(DATA_ROOT.iterdir()):
        if not run_dir.is_dir():
            continue
        run_id = run_dir.name  # e.g. 'EFD_T0001'
        csv_files = sorted(f for f in run_dir.iterdir() if f.suffix == '.csv')
        for csv_path in csv_files:
            rec = parse_filename(csv_path, run_id)
            if rec:
                records.append(rec)
    df = pd.DataFrame(records)
    return df


metadata_df = collect_metadata()
print(f'Total CSV files found: {len(metadata_df)}')
metadata_df.head()

Total CSV files found: 80


,file_path,file_name,run_id,epoch,commCh,powerStep,faultLocation,faultResistance,resistance_ohms
0,/Users/sadiazaman/Documents/EFD Challange/EFD_...,1776350717-7-1-0-0.csv,EFD_T0001,1776350717,7,1,0,0,0
1,/Users/sadiazaman/Documents/EFD Challange/EFD_...,1776351640-7-1-1-0.csv,EFD_T0001,1776351640,7,1,1,0,0
2,/Users/sadiazaman/Documents/EFD Challange/EFD_...,1776352564-7-1-1-1.csv,EFD_T0001,1776352564,7,1,1,1,5000
3,/Users/sadiazaman/Documents/EFD Challange/EFD_...,1776353487-7-1-1-2.csv,EFD_T0001,1776353487,7,1,1,2,14300
4,/Users/sadiazaman/Documents/EFD Challange/EFD_...,1776354410-7-1-1-4.csv,EFD_T0001,1776354410,7,1,1,4,30000


In [5]:
# Overview: file counts per run and fault location
print('\n=== Files per run ===')
print(metadata_df.groupby('run_id').size().rename('count'))

print('\n=== Files per (run_id, faultLocation, faultResistance) ===')
print(
    metadata_df.groupby(['run_id', 'faultLocation', 'faultResistance'])
               .size()
               .rename('count')
               .to_string()
)

print('\n=== No-fault vs faulted ===')
metadata_df['is_fault'] = metadata_df['faultLocation'] != 0
print(metadata_df.groupby(['run_id', 'is_fault']).size().rename('count'))


=== Files per run ===
run_id
EFD_T0001    24
EFD_T0002    24
EFD_T0003    32
Name: count, dtype: int64

=== Files per (run_id, faultLocation, faultResistance) ===
run_id     faultLocation  faultResistance
EFD_T0001  0              0                  4
           1              0                  1
                          1                  1
                          2                  1
                          4                  1
                          8                  1
           2              0                  1
                          1                  1
                          2                  1
                          4                  1
                          8                  1
           3              0                  1
                          1                  1
                          2                  1
                          4                  1
                          8                  1
           4              0               

## 3. CSV Parser — Four Typed DataFrames per Scenario

In [7]:
def parse_csv(file_path: Path) -> dict[str, pd.DataFrame]:
    """
    Read one scenario CSV and split rows by message type.
    Returns a dict with keys 'COMM', 'RATES', 'EXCOMM', 'CORR'.
    Malformed rows are logged but never silently dropped.
    """
    raw_rows = {'COMM': [], 'RATES': [], 'EXCOMM': [], 'CORR': []}
    malformed = []

    with open(file_path, 'r') as fh:
        for line_no, line in enumerate(fh, start=1):
            line = line.strip()
            if not line:
                continue
            parts = [p.strip() for p in line.split(',')]
            if len(parts) < 3:
                malformed.append({'line': line_no, 'reason': 'too few columns', 'content': line})
                continue
            msg_type = parts[1]
            if msg_type not in SCHEMA_MAP:
                malformed.append({'line': line_no, 'reason': f'unknown type {msg_type!r}', 'content': line})
                continue
            expected_ncols = len(SCHEMA_MAP[msg_type])
            if len(parts) != expected_ncols:
                malformed.append({
                    'line': line_no,
                    'reason': f'{msg_type} expected {expected_ncols} cols, got {len(parts)}',
                    'content': line
                })
                continue
            raw_rows[msg_type].append(parts)

    dfs = {}
    for mtype, cols in SCHEMA_MAP.items():
        if raw_rows[mtype]:
            df = pd.DataFrame(raw_rows[mtype], columns=cols)
            # Cast types
            df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
            df['remote_id'] = pd.to_numeric(df['remote_id'], errors='coerce').astype('Int64')
            for col in cols[3:]:  # all payload columns are numeric
                df[col] = pd.to_numeric(df[col], errors='coerce')
        else:
            df = pd.DataFrame(columns=cols)
        dfs[mtype] = df

    return dfs, malformed


# ── Quick smoke-test on the first file ────────────────────────────────────────
first_path = Path(metadata_df.iloc[0]['file_path'])
sample_dfs, sample_mal = parse_csv(first_path)

print(f'File: {first_path.name}')
for mtype, df in sample_dfs.items():
    print(f'  {mtype:6s}  rows={len(df):5d}  fixtures={df["remote_id"].nunique()}')
print(f'  Malformed rows: {len(sample_mal)}')

File: 1776350717-7-1-0-0.csv
  COMM    rows= 8280  fixtures=90
  RATES   rows= 8280  fixtures=90
  EXCOMM  rows= 1796  fixtures=90
  CORR    rows= 1796  fixtures=90
  Malformed rows: 0


In [8]:
# Display sample DataFrames
print('=== COMM sample ===')
display(sample_dfs['COMM'].head(3))

print('=== RATES sample ===')
display(sample_dfs['RATES'].head(3))

print('=== EXCOMM sample ===')
display(sample_dfs['EXCOMM'].head(3))

print('=== CORR sample ===')
display(sample_dfs['CORR'].head(3))

=== COMM sample ===


,timestamp,message_type,remote_id,response_rate,num_attempts,num_failures,ds_avg_repeater_depth,us_avg_repeater_depth,ds_current_repeater_depth,us_current_repeater_depth,avg_receive_signal_strength,incorrect_count
0,1.776351e+09,COMM,1001,100,11,0,0.0,0.0,0,0,0.0,0
1,1.776351e+09,COMM,1002,100,10,0,0.0,0.0,0,0,0.0,0
2,1.776351e+09,COMM,1003,100,11,0,0.0,0.0,0,0,0.0,0


=== RATES sample ===


,timestamp,message_type,remote_id,dsRawRate0,dsRawRate1,dsRawRate2,dsRawRate3,dsRawRate4,dsRawRate5,dsRawRate6,dsRawRate7,usRawRate0,usRawRate1,usRawRate2,usRawRate3,usRawRate4,usRawRate5,usRawRate6,usRawRate7
0,1.776351e+09,RATES,1001,0.999997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.999997,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.776351e+09,RATES,1002,0.999997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.999997,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.776351e+09,RATES,1003,0.999997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.999997,0.0,0.0,0.0,0.0,0.0,0.0,0.0


=== EXCOMM sample ===


,timestamp,message_type,remote_id,peakInputLevel,overflowStatus,berCount,phaseNoise_dB,rxCrcFailCount
0,1.776351e+09,EXCOMM,1001,725,0,0,-5.925146,0
1,1.776351e+09,EXCOMM,1002,673,0,0,0.585947,0
2,1.776351e+09,EXCOMM,1003,624,0,0,3.129292,0


=== CORR sample ===


,timestamp,message_type,remote_id,peakCorrLevel_dB,peakUncorrLevel_dB,ftryCorrTrigCount,userCorrTrigCount,corrFired,ftryCorrFired
0,1.776351e+09,CORR,1001,13.546875,100.273438,0,443,1,0
1,1.776351e+09,CORR,1002,13.558594,100.531250,0,489,1,0
2,1.776351e+09,CORR,1003,13.546875,99.886719,0,537,1,0


## 4. Validation — Per-File Integrity Checks

In [9]:
def validate_scenario(file_name: str, dfs: dict[str, pd.DataFrame],
                      malformed: list[dict]) -> dict:
    """
    Run integrity checks on one scenario's parsed DataFrames.
    Returns a dict of check results to be aggregated into the validation report.
    """
    rec = {'file_name': file_name, 'malformed_rows': len(malformed)}
    issues = []

    all_timestamps = []
    all_remote_ids = set()

    for mtype in ['COMM', 'RATES', 'EXCOMM', 'CORR']:
        df = dfs[mtype]
        n_rows = len(df)
        rec[f'{mtype}_rows'] = n_rows

        # ── Check 1: message type present ────────────────────────────────────
        if n_rows == 0:
            issues.append(f'MISSING message type: {mtype}')
            rec[f'{mtype}_fixtures'] = 0
            continue

        # ── Check 2: remote_id range ─────────────────────────────────────────
        ids = df['remote_id'].dropna().astype(int)
        out_of_range = ids[(ids < REMOTE_ID_MIN) | (ids > REMOTE_ID_MAX)]
        if len(out_of_range) > 0:
            issues.append(
                f'{mtype}: {len(out_of_range)} remote_id values out of [{REMOTE_ID_MIN},{REMOTE_ID_MAX}]'
            )

        # ── Check 3: all 90 fixture IDs present ──────────────────────────────
        present_ids = set(ids.unique())
        missing_ids = EXPECTED_FIXTURES - present_ids
        extra_ids   = present_ids - EXPECTED_FIXTURES
        rec[f'{mtype}_fixtures'] = len(present_ids)
        if missing_ids:
            issues.append(f'{mtype}: missing fixture IDs: {sorted(missing_ids)[:10]}' +
                          (' ...' if len(missing_ids) > 10 else ''))
        if extra_ids:
            issues.append(f'{mtype}: unexpected fixture IDs: {sorted(extra_ids)[:5]}')

        all_remote_ids.update(present_ids)
        all_timestamps.extend(df['timestamp'].dropna().tolist())

        # ── Check 4: per-fixture row count roughly as expected ────────────────
        per_fixture = df.groupby('remote_id').size()
        expected_n = EXPECTED_COUNTS[mtype]
        median_count = per_fixture.median()
        rec[f'{mtype}_median_rows_per_fixture'] = round(median_count, 1)
        # Allow wide tolerance (50–200 % of expected) for a warning only
        if median_count < expected_n * 0.5 or median_count > expected_n * 2.0:
            issues.append(
                f'{mtype}: median rows/fixture={median_count:.1f}, expected≈{expected_n}'
            )

    # ── Check 5: timestamp duration ───────────────────────────────────────────
    if all_timestamps:
        ts_min = min(all_timestamps)
        ts_max = max(all_timestamps)
        duration = ts_max - ts_min
        rec['ts_min']    = round(ts_min, 2)
        rec['ts_max']    = round(ts_max, 2)
        rec['duration_s'] = round(duration, 1)
        if abs(duration - EXPECTED_DURATION_SEC) / EXPECTED_DURATION_SEC > DURATION_WARN_TOLERANCE:
            issues.append(
                f'Duration={duration:.0f}s, expected≈{EXPECTED_DURATION_SEC}s '
                f'(tolerance {DURATION_WARN_TOLERANCE*100:.0f}%)'
            )
    else:
        rec['ts_min'] = rec['ts_max'] = rec['duration_s'] = None
        issues.append('No valid timestamps found')

    rec['issues'] = ' | '.join(issues) if issues else 'OK'
    rec['valid']  = len(issues) == 0
    return rec


print('Validation functions defined.')

Validation functions defined.


In [10]:
# ── Run validation across ALL CSV files ───────────────────────────────────────
print(f'Validating {len(metadata_df)} CSV files ...')

validation_records = []
all_malformed = {}  # file_name -> list of malformed rows

for idx, row in metadata_df.iterrows():
    fp = Path(row['file_path'])
    try:
        dfs, malformed = parse_csv(fp)
    except Exception as e:
        validation_records.append({
            'file_name': row['file_name'],
            'valid': False,
            'issues': f'PARSE ERROR: {e}'
        })
        continue

    val_rec = validate_scenario(row['file_name'], dfs, malformed)
    validation_records.append(val_rec)

    if malformed:
        all_malformed[row['file_name']] = malformed

    if (idx + 1) % 20 == 0:
        print(f'  {idx + 1}/{len(metadata_df)} done')

print(f'Done. {len(validation_records)} records created.')

validation_df = pd.DataFrame(validation_records)
print(f"\nValid files : {validation_df['valid'].sum()}")
print(f"Invalid files: {(~validation_df['valid']).sum()}")

Validating 80 CSV files ...
  20/80 done
  40/80 done
  60/80 done
  80/80 done
Done. 80 records created.

Valid files : 80
Invalid files: 0


In [11]:
# Show any files with issues
invalid = validation_df[~validation_df['valid']]
if len(invalid) > 0:
    print('=== Files with validation issues ===')
    display(invalid[['file_name', 'issues']].to_string(index=False))
else:
    print('All files passed validation.')

# Show the full validation report
display(validation_df.head(10))

All files passed validation.


,file_name,malformed_rows,COMM_rows,COMM_fixtures,COMM_median_rows_per_fixture,RATES_rows,RATES_fixtures,RATES_median_rows_per_fixture,EXCOMM_rows,EXCOMM_fixtures,EXCOMM_median_rows_per_fixture,CORR_rows,CORR_fixtures,CORR_median_rows_per_fixture,ts_min,ts_max,duration_s,issues,valid
0,1776350717-7-1-0-0.csv,0,8280,90,92.0,8280,90,92.0,1796,90,20.0,1796,90,20.0,1.776351e+09,1.776352e+09,916.2,OK,True
1,1776351640-7-1-1-0.csv,0,8280,90,92.0,8280,90,92.0,1858,90,21.0,1858,90,21.0,1.776352e+09,1.776353e+09,917.5,OK,True
2,1776352564-7-1-1-1.csv,0,8280,90,92.0,8280,90,92.0,1867,90,21.0,1866,90,21.0,1.776353e+09,1.776353e+09,917.3,OK,True
3,1776353487-7-1-1-2.csv,0,8370,90,93.0,8370,90,93.0,1854,90,21.0,1853,90,21.0,1.776353e+09,1.776354e+09,920.4,OK,True
4,1776354410-7-1-1-4.csv,0,8280,90,92.0,8280,90,92.0,1868,90,21.0,1869,90,21.0,1.776354e+09,1.776355e+09,917.6,OK,True
5,1776355334-7-1-1-8.csv,0,8280,90,92.0,8280,90,92.0,1883,90,21.0,1884,90,21.0,1.776355e+09,1.776356e+09,917.5,OK,True
6,1776356257-7-1-0-0.csv,0,8370,90,93.0,8370,90,93.0,1852,90,21.0,1851,90,21.0,1.776356e+09,1.776357e+09,920.6,OK,True
7,1776357180-7-1-2-0.csv,0,8280,90,92.0,8280,90,92.0,1862,90,21.0,1860,90,21.0,1.776357e+09,1.776358e+09,917.6,OK,True
8,1776358104-7-1-2-1.csv,0,8280,90,92.0,8280,90,92.0,1857,90,21.0,1856,90,21.0,1.776358e+09,1.776359e+09,917.6,OK,True
9,1776359027-7-1-2-2.csv,0,8370,90,93.0,8370,90,93.0,1872,90,21.0,1871,90,21.0,1.776359e+09,1.776360e+09,920.2,OK,True


In [12]:
# ── Row-count summary by message type ────────────────────────────────────────
print('=== Median rows per fixture by message type (across all files) ===')
for mtype in ['COMM', 'RATES', 'EXCOMM', 'CORR']:
    col = f'{mtype}_median_rows_per_fixture'
    if col in validation_df.columns:
        print(f'  {mtype:6s}: median={validation_df[col].median():.1f}, '
              f'min={validation_df[col].min():.1f}, '
              f'max={validation_df[col].max():.1f}  '
              f'(expected≈{EXPECTED_COUNTS[mtype]})')

print('\n=== Duration statistics (seconds) ===')
if 'duration_s' in validation_df.columns:
    dur = validation_df['duration_s'].dropna()
    print(f'  mean={dur.mean():.1f}  min={dur.min():.1f}  '
          f'max={dur.max():.1f}  expected≈{EXPECTED_DURATION_SEC}')

=== Median rows per fixture by message type (across all files) ===
  COMM  : median=92.0, min=92.0, max=93.0  (expected≈92)
  RATES : median=92.0, min=92.0, max=93.0  (expected≈92)
  EXCOMM: median=21.0, min=18.0, max=21.0  (expected≈20)
  CORR  : median=21.0, min=18.0, max=21.0  (expected≈20)

=== Duration statistics (seconds) ===
  mean=918.3  min=913.0  max=920.6  expected≈900


In [13]:
# ── Log malformed rows ────────────────────────────────────────────────────────
total_malformed = sum(len(v) for v in all_malformed.values())
print(f'Total malformed rows across all files: {total_malformed}')
if all_malformed:
    print('\nFiles with malformed rows:')
    for fname, rows in all_malformed.items():
        print(f'  {fname}: {len(rows)} malformed row(s)')
        for r in rows[:3]:
            print(f'    line {r["line"]}: {r["reason"]}  → {r["content"][:80]}')

Total malformed rows across all files: 0


## 5. Save Outputs

In [14]:
# ── Merge validation stats back into metadata ─────────────────────────────────
# We keep metadata and validation separate; the metadata table never carries
# faultLocation/faultResistance into the prediction path.
meta_out_path = OUTPUTS_DIR / 'metadata.csv'
val_out_path  = OUTPUTS_DIR / 'validation_report.csv'

metadata_df.to_csv(meta_out_path, index=False)
print(f'Saved metadata        → {meta_out_path}')

# Join validation report with run_id from metadata for context
val_export = validation_df.merge(
    metadata_df[['file_name', 'run_id', 'faultLocation', 'faultResistance']],
    on='file_name', how='left'
)
val_export.to_csv(val_out_path, index=False)
print(f'Saved validation report → {val_out_path}')

Saved metadata        → /Users/sadiazaman/Documents/EFD Challange/outputs/metadata.csv
Saved validation report → /Users/sadiazaman/Documents/EFD Challange/outputs/validation_report.csv


## 6. Dataset Overview Summary

In [15]:
print('=' * 60)
print('DATASET SUMMARY')
print('=' * 60)
print(f'Total scenario files : {len(metadata_df)}')
print(f'Runs                 : {sorted(metadata_df["run_id"].unique())}')
print(f'Fault locations      : {sorted(metadata_df["faultLocation"].unique())}  (0=no fault)')
print(f'Resistance codes     : {sorted(metadata_df["faultResistance"].unique())}')
print()
print('Files per run and fault location:')
pivot = (
    metadata_df
    .groupby(['run_id', 'faultLocation'])
    .size()
    .unstack(fill_value=0)
)
print(pivot.to_string())
print()
print(f'Validation: {validation_df["valid"].sum()} / {len(validation_df)} files passed all checks')
print('=' * 60)

DATASET SUMMARY
Total scenario files : 80
Runs                 : ['EFD_T0001', 'EFD_T0002', 'EFD_T0003']
Fault locations      : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]  (0=no fault)
Resistance codes     : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(8)]

Files per run and fault location:
faultLocation  0  1  2  3  4
run_id                      
EFD_T0001      4  5  5  5  5
EFD_T0002      4  5  5  5  5
EFD_T0003      4  7  7  7  7

Validation: 80 / 80 files passed all checks


In [16]:
# ── Show per-run fixture ID coverage (from a sample file per run) ─────────────
print('Remote ID coverage check (sample file per run):')
for run_id in sorted(metadata_df['run_id'].unique()):
    sample_row = metadata_df[metadata_df['run_id'] == run_id].iloc[0]
    dfs, _ = parse_csv(Path(sample_row['file_path']))
    for mtype in ['COMM', 'RATES', 'EXCOMM', 'CORR']:
        ids = set(dfs[mtype]['remote_id'].dropna().astype(int).unique())
        missing = EXPECTED_FIXTURES - ids
        print(f'  {run_id} / {mtype:6s}: {len(ids)} fixtures present, '
              f'missing={sorted(missing) if missing else "none"}')

Remote ID coverage check (sample file per run):
  EFD_T0001 / COMM  : 90 fixtures present, missing=none
  EFD_T0001 / RATES : 90 fixtures present, missing=none
  EFD_T0001 / EXCOMM: 90 fixtures present, missing=none
  EFD_T0001 / CORR  : 90 fixtures present, missing=none
  EFD_T0002 / COMM  : 90 fixtures present, missing=none
  EFD_T0002 / RATES : 90 fixtures present, missing=none
  EFD_T0002 / EXCOMM: 90 fixtures present, missing=none
  EFD_T0002 / CORR  : 90 fixtures present, missing=none
  EFD_T0003 / COMM  : 90 fixtures present, missing=none
  EFD_T0003 / RATES : 90 fixtures present, missing=none
  EFD_T0003 / EXCOMM: 90 fixtures present, missing=none
  EFD_T0003 / CORR  : 90 fixtures present, missing=none


## 7. Sample DataFrames — One Scenario

Display the four typed DataFrames for the first scenario file as a reference for downstream notebooks.

In [17]:
# Reload the first file as a clean reference
ref_path = Path(metadata_df.iloc[0]['file_path'])
ref_dfs, _ = parse_csv(ref_path)

print(f'Reference file: {ref_path.name}')
print()
for mtype in ['COMM', 'RATES', 'EXCOMM', 'CORR']:
    df = ref_dfs[mtype]
    print(f'--- {mtype} ({len(df)} rows, {df["remote_id"].nunique()} fixtures) ---')
    display(df.describe().round(3))
    print()

Reference file: 1776350717-7-1-0-0.csv

--- COMM (8280 rows, 90 fixtures) ---


,timestamp,remote_id,response_rate,num_attempts,num_failures,ds_avg_repeater_depth,us_avg_repeater_depth,ds_current_repeater_depth,us_current_repeater_depth,avg_receive_signal_strength,incorrect_count
count,8.280000e+03,8280.0,8280.000,8280.000,8280.000,8280.0,8280.0,8280.0,8280.0,8280.0,8280.0
mean,1.776351e+09,1045.5,99.994,370.133,0.077,0.0,0.0,0.0,0.0,0.0,0.0
std,2.655920e+02,25.981,0.077,212.365,0.269,0.0,0.0,0.0,0.0,0.0,0.0
min,1.776351e+09,1001.0,99.000,6.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0
25%,1.776351e+09,1023.0,100.000,188.750,0.000,0.0,0.0,0.0,0.0,0.0,0.0
50%,1.776351e+09,1045.5,100.000,368.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0
75%,1.776351e+09,1068.0,100.000,552.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0
max,1.776352e+09,1090.0,100.000,742.000,2.000,0.0,0.0,0.0,0.0,0.0,0.0



--- RATES (8280 rows, 90 fixtures) ---


,timestamp,remote_id,dsRawRate0,dsRawRate1,dsRawRate2,dsRawRate3,dsRawRate4,dsRawRate5,dsRawRate6,dsRawRate7,usRawRate0,usRawRate1,usRawRate2,usRawRate3,usRawRate4,usRawRate5,usRawRate6,usRawRate7
count,8.280000e+03,8280.0,8280.000,8280.0,8280.0,8280.0,8280.0,8280.0,8280.0,8280.0,8280.000,8280.0,8280.0,8280.0,8280.0,8280.0,8280.0,8280.0
mean,1.776351e+09,1045.5,0.999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.999,0.0,0.0,0.0,0.0,0.0,0.0,0.0
std,2.655920e+02,25.981,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0
min,1.776351e+09,1001.0,0.984,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.984,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,1.776351e+09,1023.0,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50%,1.776351e+09,1045.5,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
75%,1.776351e+09,1068.0,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,1.776352e+09,1090.0,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0



--- EXCOMM (1796 rows, 90 fixtures) ---


,timestamp,remote_id,peakInputLevel,overflowStatus,berCount,phaseNoise_dB,rxCrcFailCount
count,1.796000e+03,1796.0,1796.000,1796.000,1796.000,1796.000,1796.000
mean,1.776351e+09,1045.42,843.252,0.428,4.693,0.124,0.158
std,2.634860e+02,25.953,301.933,5.215,29.419,4.173,1.123
min,1.776351e+09,1001.0,0.000,0.000,0.000,-16.532,0.000
25%,1.776351e+09,1023.0,623.000,0.000,0.000,-2.382,0.000
50%,1.776351e+09,1045.0,782.000,0.000,0.000,0.988,0.000
75%,1.776351e+09,1068.0,1153.250,0.000,0.000,3.402,0.000
max,1.776352e+09,1090.0,1355.000,64.000,343.000,8.840,14.000



--- CORR (1796 rows, 90 fixtures) ---


,timestamp,remote_id,peakCorrLevel_dB,peakUncorrLevel_dB,ftryCorrTrigCount,userCorrTrigCount,corrFired,ftryCorrFired
count,1.796000e+03,1796.0,1796.000,1796.000,1796.0,1796.000,1796.000,1796.0
mean,1.776351e+09,1045.413,13.612,101.318,0.0,27544.246,0.999,0.0
std,2.634940e+02,25.948,0.326,3.917,0.0,18994.708,0.024,0.0
min,1.776351e+09,1001.0,0.000,18.000,0.0,0.000,0.000,0.0
25%,1.776351e+09,1023.0,13.605,99.168,0.0,11211.750,1.000,0.0
50%,1.776351e+09,1045.0,13.641,101.373,0.0,22756.500,1.000,0.0
75%,1.776351e+09,1068.0,13.652,104.487,0.0,43747.250,1.000,0.0
max,1.776352e+09,1090.0,13.711,106.812,0.0,65499.000,1.000,0.0


## 8. Loader Utility — Exported for Downstream Notebooks

The `parse_csv` function and schema constants defined in this notebook are the canonical parsing interface. Downstream notebooks import them via `%run` or by copying the relevant cells. The loader's contract:

- Input: a `Path` to a scenario CSV
- Output: `dict[str, pd.DataFrame]` with keys `COMM`, `RATES`, `EXCOMM`, `CORR`, plus a list of malformed row records
- All numeric columns cast to float/int; `remote_id` always `Int64`
- Fixture order must be restored by sorting on `remote_id` before any spatial operation

In [18]:
print('Notebook 01 complete.')
print(f'  outputs/metadata.csv          : {meta_out_path}')
print(f'  outputs/validation_report.csv : {val_out_path}')

Notebook 01 complete.
  outputs/metadata.csv          : /Users/sadiazaman/Documents/EFD Challange/outputs/metadata.csv
  outputs/validation_report.csv : /Users/sadiazaman/Documents/EFD Challange/outputs/validation_report.csv
